## Release the trained model to the Hugging Face Hub

Persists the weights so you never retrain just to evaluate: after this, `--eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b` loads the model from anywhere, on any test file, with zero training. Run these cells only after you're happy with the results above. You need a Hugging Face **write** token (https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a WRITE token

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
REPO_ID = "esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b"
SEED = 7  # which trained checkpoint to release (seed 7 = representative)
src = f"ckpt_ce_seed{SEED}"
# save in fp16 to halve the upload (~1 GB); eval loads it fine
m = AutoModelForCausalLM.from_pretrained(src, torch_dtype=torch.float16)
t = AutoTokenizer.from_pretrained(src)
m.push_to_hub(REPO_ID); t.push_to_hub(REPO_ID)
# upload the model card if present in the cloned repo
import os
if os.path.exists('MODEL_CARD.md'):
    from huggingface_hub import upload_file
    upload_file(path_or_fileobj='MODEL_CARD.md', path_in_repo='README.md', repo_id=REPO_ID)
print('released to https://huggingface.co/' + REPO_ID)

In [ ]:
# sanity: load the RELEASED model from the Hub and score the committed test
!PYTHONPATH=. python train_verifier_reward.py \
  --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file benchmark_test.jsonl

# Verifier cross-entropy fine-tuning (Qwen2.5-0.5B) — the convergent run

The RL objectives fail by optimization pathology: sampled REINFORCE
collapses (variance), exact expected-reward PG oscillates (its gradient
~ pi(A)pi(R) saturates at the corners). **Verifier cross-entropy** trains
`-log pi(verdict)`; its gradient stays strong when the model is
confidently wrong, so it converges. Still verifier-only supervision (the
verdict is the sole signal), now used as a target instead of a scalar
reward, and trained on the **same natural-language prompts** the frontier
ladder is scored on so the final comparison is fair.

**Leakage discipline unchanged**: train = expanded_train (seed 101),
monitor = expanded_val (seed 202), both deduplicated against the committed
benchmark_test.jsonl, which is evaluated ONCE at the end.

Runtime → Change runtime type → **L4 GPU** (or T4) → Save → Run all.
**Download `ce_results.zip` the moment the last cell finishes.**

In [ ]:
# Idempotent clone + absolute cd (prevents nested-directory issues).
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
print('cwd:', os.getcwd())
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
!PYTHONPATH=. python test_authority_verifier.py


In [ ]:
!PYTHONPATH=. python make_expanded_train.py \
  --train-seed 101 --train-traces-per-class 150 \
  --val-seed 202 --val-traces-per-class 25

In [ ]:
# Verifier cross-entropy, NL prompts, balanced, 3 seeds. Curves on VAL.
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --ce-loss --balance-reward --prompt-style nl \
    --steps 500 --batch-size 16 --lr 2e-5 \
    --train-file expanded_train.jsonl --test-file expanded_val.jsonl \
    --eval-every 50 --eval-max-actions 120 \
    --seed $s --log-file training_log_qwen05b_ce_seed$s.jsonl \
    --save-dir ckpt_ce_seed$s || break ; done

In [ ]:
# FINAL evaluation — committed held-out test set, natural-language prompts
# identical to training and to the API-model ladder. Run once.
import json
json.dump({"backends": {}}, open("results_ce.json", "w"))
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py \
    --eval-checkpoint ckpt_ce_seed$s --test-file benchmark_test.jsonl \
    --merge-results results_ce.json || break ; done
print(json.dumps(json.load(open("results_ce.json")), indent=2)[:1500])

In [ ]:
# Optional keep-weights: mount Drive and copy a checkpoint before download.
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r ckpt_ce_seed7 /content/drive/MyDrive/
!zip -j ce_results.zip training_log_qwen05b_ce_seed*.jsonl results_ce.json
from google.colab import files
files.download("ce_results.zip")